In [6]:
import numpy as np

# =========================================================
# 1) Dataset (3 sensores digitales -> 2 PWM normalizados)
#    sL,sC,sR: 1 = negro (línea), 0 = blanco
#    yL,yR: duty 0..1 para motor izquierdo/derecho
# =========================================================
X = np.array([
    [0,0,0],
    [0,0,1],
    [0,1,0],
    [0,1,1],
    [1,0,0],
    [1,0,1],
    [1,1,0],
    [1,1,1],
], dtype=np.float32)

Y = np.array([
    [0.80, 0.15],  # 000: perdido -> buscar derecha
    [0.95, 0.10],  # 001: giro fuerte derecha
    [0.85, 0.85],  # 010: recto
    [0.90, 0.45],  # 011: giro suave derecha
    [0.10, 0.95],  # 100: giro fuerte izquierda
    [0.55, 0.55],  # 101: ambiguo -> cautela
    [0.45, 0.90],  # 110: giro suave izquierda
    [0.60, 0.60],  # 111: intersección -> recto lento
], dtype=np.float32)

# =========================================================
# 2) Activación sigmoide (estable)
# =========================================================
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.clip(z, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-z))

def d_sigmoid(a: np.ndarray) -> np.ndarray:
    # derivada en función de la activación
    return a * (1.0 - a)

# =========================================================
# 3) MLP: 3 -> 4 -> 2 (full-batch GD)
# =========================================================
np.random.seed(42)
n_in, n_h, n_out = 3, 4, 2

W1 = (np.random.randn(n_in, n_h).astype(np.float32) * 0.8)
b1 = np.zeros((1, n_h), dtype=np.float32)
W2 = (np.random.randn(n_h, n_out).astype(np.float32) * 0.8)
b2 = np.zeros((1, n_out), dtype=np.float32)

eta = 0.6
epochs = 20000
print_every = 2000  # evita output enorme (y truncamiento)

for ep in range(1, epochs + 1):
    # forward
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    yhat = sigmoid(z2)

    # loss MSE
    loss = np.mean((Y - yhat) ** 2)

    # backprop
    dZ2 = (yhat - Y) * d_sigmoid(yhat)
    dW2 = (a1.T @ dZ2) / len(X)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / len(X)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * d_sigmoid(a1)
    dW1 = (X.T @ dZ1) / len(X)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / len(X)

    # update
    W2 -= eta * dW2
    b2 -= eta * db2
    W1 -= eta * dW1
    b1 -= eta * db1

    if print_every and (ep % print_every == 0):
        print(f"Epoch {ep:5d} | Loss={loss:.6f}")

# =========================================================
# 4) Verificación rápida (8 casos base)
# =========================================================
a1 = sigmoid(X @ W1 + b1)
yhat = sigmoid(a1 @ W2 + b2)

print("\nX -> target vs pred (redondeado):")
for xi, yi, pi in zip(X, Y, yhat):
    print(xi.astype(int), " target=", np.round(yi, 2), " pred=", np.round(pi, 2))

# =========================================================
# 5) Export a Arduino SIN truncado
#    Convención A:
#      W1[in][hid] = [3][4]
#      W2[hid][out] = [4][2]
#
#    Si quieres el estilo de tu imagen:
#      W1[hid][in] = [4][3]  (W1.T)
#      W2[out][hid] = [2][4] (W2.T)
#    Pon EXPORT_STYLE = "B"
# =========================================================
EXPORT_STYLE = "A"  # "A" o "B"

def c_array_2d(name: str, A: np.ndarray) -> str:
    A = np.asarray(A, dtype=np.float32)
    r, c = A.shape
    lines = [f"const float {name}[{r}][{c}] = {{"]
    for i in range(r):
        row = ", ".join(f"{A[i, j]: .8f}f" for j in range(c))
        lines.append(f"  {{{row}}},")
    lines.append("};\n")
    return "\n".join(lines)

def c_array_1d(name: str, v: np.ndarray) -> str:
    v = np.asarray(v, dtype=np.float32).reshape(-1)
    items = ", ".join(f"{x: .8f}f" for x in v)
    return f"const float {name}[{v.size}] = {{{items}}};\n"

if EXPORT_STYLE.upper() == "A":
    W1_exp = W1          # [3][4]
    W2_exp = W2          # [4][2]
    style_note = "// Style A: W1[3][4] (in->hid), W2[4][2] (hid->out)\n"
else:
    W1_exp = W1.T        # [4][3]
    W2_exp = W2.T        # [2][4]
    style_note = "// Style B: W1[4][3] (hid->in), W2[2][4] (out->hid)\n"

export_text = []
export_text.append("// ====== ARDUINO EXPORT ======\n")
export_text.append(style_note)
export_text.append(c_array_2d("W1", W1_exp))
export_text.append(c_array_1d("b1", b1))
export_text.append(c_array_2d("W2", W2_exp))
export_text.append(c_array_1d("b2", b2))

export_text = "\n".join(export_text)

# Escribe a archivo para copiar/pegar sin '...'
out_file_h = "mlp_weights_arduino.h"
out_file_txt = "mlp_weights_arduino.txt"
with open(out_file_h, "w", encoding="utf-8") as f:
    f.write(export_text)
with open(out_file_txt, "w", encoding="utf-8") as f:
    f.write(export_text)

print(f"\nListo. Pesos guardados en:\n- {out_file_h}\n- {out_file_txt}")
print("Copia desde el archivo (no desde la consola) para evitar '...'.")

Epoch  2000 | Loss=0.004528
Epoch  4000 | Loss=0.001682
Epoch  6000 | Loss=0.001073
Epoch  8000 | Loss=0.000808
Epoch 10000 | Loss=0.000644
Epoch 12000 | Loss=0.000526
Epoch 14000 | Loss=0.000432
Epoch 16000 | Loss=0.000355
Epoch 18000 | Loss=0.000293
Epoch 20000 | Loss=0.000246

X -> target vs pred (redondeado):
[0 0 0]  target= [0.8  0.15]  pred= [0.81 0.16]
[0 0 1]  target= [0.95 0.1 ]  pred= [0.91 0.06]
[0 1 0]  target= [0.85 0.85]  pred= [0.85 0.85]
[0 1 1]  target= [0.9  0.45]  pred= [0.9  0.45]
[1 0 0]  target= [0.1  0.95]  pred= [0.09 0.92]
[1 0 1]  target= [0.55 0.55]  pred= [0.55 0.55]
[1 1 0]  target= [0.45 0.9 ]  pred= [0.44 0.9 ]
[1 1 1]  target= [0.6 0.6]  pred= [0.61 0.61]

Listo. Pesos guardados en:
- mlp_weights_arduino.h
- mlp_weights_arduino.txt
Copia desde el archivo (no desde la consola) para evitar '...'.
